# WP5 — Baselines starter (Student 2)

**What this notebook does.** Reads `probability_table.parquet` and `labels.parquet`, emits three fixed-rule baselines (`fixed_3`, `fixed_5`, `fixed_7`) and an `always_predict` baseline per pipeline contract §5. Also seeds `tau_per_strategy.parquet`.

Oracle-global and signal-quality baselines are left as TODO stubs — they require training-time optimisation that is Student 2's WP5 work.

**Outputs.**
- `decisions/fixed_3.parquet`, `fixed_5.parquet`, `fixed_7.parquet`, `always_predict.parquet`
- `tau_per_strategy.parquet` (seeded; WP6 appends later)


In [ ]:
import sys, os, subprocess
from pathlib import Path

REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import numpy as np
import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
except ImportError:
    pass

DATA = Path('synthetic/v1')
DATA.mkdir(parents=True, exist_ok=True)
(DATA / 'preprocessing').mkdir(exist_ok=True)
(DATA / 'endpoint').mkdir(exist_ok=True)
(DATA / 'decisions').mkdir(exist_ok=True)
(DATA / 'evaluation').mkdir(exist_ok=True)
print(f'Pipeline root: {DATA}')


## 1. Load inputs


In [ ]:
prob = pd.read_parquet(DATA / 'probability_table.parquet')
lbls = pd.read_parquet(DATA / 'labels.parquet')
print(f'{len(prob):,} predictions, {prob["participant_id"].nunique()} participants')


## 2. Helper — emit a fixed-K baseline


In [ ]:
def fixed_k_decisions(prob: pd.DataFrame, k: int) -> pd.DataFrame:
    '''Defer until night_index < k, then predict once at night_index == k.'''
    rows = []
    for pid, g in prob.sort_values(['participant_id','night_index']).groupby('participant_id'):
        for _, r in g.iterrows():
            if r['night_index'] < k:
                rows.append({'participant_id': pid, 'night_index': int(r['night_index']),
                             'cumulative_k': int(r['cumulative_k']),
                             'strategy_name': f'fixed_{k}', 'decision': 'defer',
                             'predicted_label': None, 'prediction_set_size': None,
                             'mondrian_stratum': None, 'is_synthetic': bool(r['is_synthetic'])})
            elif r['night_index'] == k:
                predicted = 'post' if r['p_post_ovulatory'] >= 0.5 else 'pre'
                rows.append({'participant_id': pid, 'night_index': int(r['night_index']),
                             'cumulative_k': int(r['cumulative_k']),
                             'strategy_name': f'fixed_{k}', 'decision': 'predict',
                             'predicted_label': predicted, 'prediction_set_size': None,
                             'mondrian_stratum': None, 'is_synthetic': bool(r['is_synthetic'])})
                break  # fixed-rule commits once and stops
    df = pd.DataFrame(rows)
    return df.astype({
        'participant_id': 'string', 'night_index': 'int32', 'cumulative_k': 'int32',
        'strategy_name': 'string', 'decision': 'string',
        'predicted_label': 'string', 'prediction_set_size': 'Int8',
        'mondrian_stratum': 'string', 'is_synthetic': 'bool',
    })


## 3. Emit fixed_3, fixed_5, fixed_7


In [ ]:
for k in [3, 5, 7]:
    dec = fixed_k_decisions(prob, k)
    dec.to_parquet(DATA / f'decisions/fixed_{k}.parquet', index=False)
    n_predicted = (dec['decision'] == 'predict').sum()
    print(f'fixed_{k}: {len(dec):,} rows, {n_predicted} predictions (expected {prob["participant_id"].nunique()})')


## 4. Emit always_predict


In [ ]:
ap = prob.copy()
ap['strategy_name'] = 'always_predict'
ap['decision'] = 'predict'
ap['predicted_label'] = np.where(ap['p_post_ovulatory'] >= 0.5, 'post', 'pre')
ap['prediction_set_size'] = pd.array([pd.NA] * len(ap), dtype='Int8')
ap['mondrian_stratum'] = pd.Series([None] * len(ap), dtype='string')
ap = ap[['participant_id','night_index','cumulative_k','strategy_name',
         'decision','predicted_label','prediction_set_size','mondrian_stratum','is_synthetic']]
ap = ap.astype({'participant_id': 'string', 'strategy_name': 'string',
                'decision': 'string', 'predicted_label': 'string', 'mondrian_stratum': 'string'})
ap.to_parquet(DATA / 'decisions/always_predict.parquet', index=False)
peek(DATA / 'decisions/always_predict.parquet')


## 5. Emit `tau_per_strategy` (fixed-rule rows)


In [ ]:
pids = sorted(prob['participant_id'].unique())
tau_rows = []
for k in [3, 5, 7]:
    for pid in pids:
        tau_rows.append({'participant_id': pid, 'strategy_name': f'fixed_{k}',
                         'tau_i': k, 'convergence_status': 'converged', 'is_synthetic': True})
for pid in pids:
    tau_rows.append({'participant_id': pid, 'strategy_name': 'always_predict',
                     'tau_i': 1, 'convergence_status': 'converged', 'is_synthetic': True})
tau = pd.DataFrame(tau_rows).astype({
    'participant_id': 'string', 'strategy_name': 'string',
    'tau_i': 'Int32', 'convergence_status': 'string', 'is_synthetic': 'bool'})
tau.to_parquet(DATA / 'tau_per_strategy.parquet', index=False)
peek(DATA / 'tau_per_strategy.parquet')


## 6. TODO — Oracle global threshold and signal-quality heuristic

These require learning on training data. Pseudocode:

```python
# oracle_global: find tau* minimising expected loss on training participants
for tau_candidate in range(1, max_nights + 1):
    loss = sum(per_participant_expected_loss(prob, lbls, tau_candidate))
    # pick argmin; emit decisions with that tau*

# signal_quality: learn threshold on mean nocturnal temp variance
threshold = train_threshold(covariates['mean_snr_temp'], train_labels)
# predict at first night where nocturnal_snr_temp >= threshold
```

Left to Student 2 once the LOSO calibration recipe (pipeline contract §11 item 4) is frozen.
